Imports

In [0]:
%pip install statsmodels seaborn

In [0]:
import sys, os
sys.path.append(os.path.abspath("..")) 


from src.mstl import SeasonalTrendDecomp
from src.isolation_forest import IsolationForestAnomalyDetector
from src.pca import PCAAnomalyDetector

from sklearn.model_selection import train_test_split 

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow



Call tables

In [0]:
cols = ["timestamp", "nd", "embedded_solar_generation","embedded_wind_generation", "year", "month","weekday", "half_hour_intervals"]

pdf = spark.table("elec_fin").select(cols).orderBy("timestamp").toPandas()
pdf = pdf.set_index("timestamp")

print(pdf.shape) # expect (105168, 7)
print(pdf.index[:3]) # expect 3 timestamps from 1 Jan 2018

MSTL - call on trend decomposer from SRC file

In [0]:
decomp = SeasonalTrendDecomp(pdf["nd"])
seasonal, trend, resid = decomp.mstl_model_train()

# outputs of columns as a result of trend decomposition 
mstl_output = pd.concat([trend, seasonal, resid], axis = 1)
# trend + seasonal + resid = nd

model_df = pdf.join(mstl_output)
print(model_df.shape)


In [0]:
# create a new dataset with the merged info of MSTl
model_df.attrs = {} # drop the hidden Spark metadata that came with toPandas()
(spark.createDataFrame(model_df.reset_index())
      .write.mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("elec_mstl"))

spark.table("elec_mstl").printSchema()


In [0]:
model_df = (spark.table("elec_mstl")
                 .orderBy("timestamp")
                 .toPandas()
                 .set_index("timestamp"))

model_df.attrs = {} # clear the Spark metadata again
model_df.to_csv("/Volumes/workspace/default/elec_raw/model_df.csv")

Train and test splitting

In [0]:
# read file in
model_df = pd.read_csv("/Volumes/workspace/default/elec_raw/model_df.csv", index_col="timestamp", parse_dates=True)


In [0]:
features = ['resid']
x_train, x_remain = train_test_split(model_df[features], test_size=0.30, random_state=42)
x_val, x_test = train_test_split(x_remain, test_size=0.50, random_state=42)

print(x_train.shape, x_val.shape, x_test.shape)

Isolation Forest


In [0]:
for d in (x_train, x_val, x_test):
    d.attrs = {}         # clear the hidden Spark metadata again, to be safe

with mlflow.start_run(run_name="isolation_forest"):
    if_ad = IsolationForestAnomalyDetector(x_train, x_val, x_test,
                                           contamination="auto",
                                           n_estimators=100)
    if_ad.train()

    _, train_raw, _, train_scores = if_ad.evaluate_train()
    val_preds, val_raw, val_decision, val_scores = if_ad.validate()

    # params: the settings that define this run
    mlflow.log_params(if_ad.model.get_params())
    mlflow.log_param("features", list(x_train.columns))
    mlflow.log_param("n_features", x_train.shape[1])

    # metrics: train and val side by side
    mlflow.log_metrics({
        "train_rows": train_scores["rows"],
        "train_anomalies": train_scores["n_anomalies"],
        "train_anomaly_rate": train_scores["anomaly_rate"],
        "train_score_median": train_scores["score_median"],
        "val_rows": val_scores["rows"],
        "val_anomalies": val_scores["n_anomalies"],
        "val_anomaly_rate": val_scores["anomaly_rate"],
        "val_score_min": val_scores["score_min"],
        "val_score_median": val_scores["score_median"],
        "val_score_max": val_scores["score_max"],
        "offset": if_ad.model.offset_,
    })

    scores, fig = if_ad.plot_score_distribution(x_val)
    mlflow.log_figure(fig, "val_score_distribution.png")
    mlflow.sklearn.log_model(if_ad.model, name="isolation_forest",
                             skops_trusted_types=["sklearn.tree._tree.Tree"])

In [0]:
X = model_df[x_train.columns] # the same 11 columns the model was trained on

model_df["anomaly_score"] = if_ad.model.score_samples(X) # lower = more anomalous
model_df["is_anomaly"] = if_ad.model.predict(X) == -1 # True / False

model_df.attrs = {} # clear the Spark metadata before saving
(spark.createDataFrame(model_df.reset_index())
      .write.mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("elec_anomalies"))

display(spark.table("elec_anomalies").orderBy("anomaly_score"))

display(spark.table("elec_anomalies").groupBy("is_anomaly").count())